In [ ]:
!pip -q install datasets transformers accelerate sentencepiece

In [ ]:
import os, re, json, gzip, itertools, subprocess, textwrap
from pathlib import Path
from collections import Counter

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline

# ----------------------------
# 1. 下载 MedMentions ST21pv 原始 PubTator 文件
# ----------------------------
DATA_DIR = Path("/content/medmentions_demo")
DATA_DIR.mkdir(exist_ok=True)

url = "https://raw.githubusercontent.com/chanzuckerberg/MedMentions/master/st21pv/data/corpus_pubtator.txt.gz"
gz_path = DATA_DIR / "corpus_pubtator.txt.gz"

if not gz_path.exists():
    !wget -q -O /content/medmentions_demo/corpus_pubtator.txt.gz https://raw.githubusercontent.com/chanzuckerberg/MedMentions/master/st21pv/data/corpus_pubtator.txt.gz

print("Downloaded:", gz_path, "size:", gz_path.stat().st_size)

Downloaded: /content/medmentions_demo/corpus_pubtator.txt.gz size: 5098267


In [ ]:
def read_first_pubtator_doc(gz_file):
    with gzip.open(gz_file, "rt", encoding="utf-8") as f:
        block = []
        for line in f:
            line = line.rstrip("\n")
            if line == "":
                if block:
                    break
            else:
                block.append(line)

    title = ""
    abstract = ""
    mentions = []
    pmid = None

    for line in block:
        if "|t|" in line:
            pmid, _, title = line.partition("|t|")
        elif "|a|" in line:
            pmid2, _, abstract = line.partition("|a|")
            pmid = pmid or pmid2
        else:
            parts = line.split("\t")
            if len(parts) >= 6:
                mentions.append({
                    "pmid": parts[0],
                    "start": int(parts[1]),
                    "end": int(parts[2]),
                    "mention": parts[3],
                    "semantic_type": parts[4],
                    "cui": parts[5],
                })

    text = title + " " + abstract
    return {
        "pmid": pmid,
        "title": title,
        "abstract": abstract,
        "text": text,
        "gold_mentions": mentions,
    }

doc = read_first_pubtator_doc(gz_path)

print("PMID:", doc["pmid"])
print("TEXT length:", len(doc["text"]))
print("Gold mention count:", len(doc["gold_mentions"]))
print("\nTEXT:\n", doc["text"][:1500])
print("\nFirst 10 gold mentions:")
for m in doc["gold_mentions"][:10]:
    print(m)

PMID: 25763772
TEXT length: 1809
Gold mention count: 66

TEXT:
 DCTN4 as a modifier of chronic Pseudomonas aeruginosa infection in cystic fibrosis Pseudomonas aeruginosa (Pa) infection in cystic fibrosis (CF) patients is associated with worse long-term pulmonary disease and shorter survival, and chronic Pa infection (CPA) is associated with reduced lung function, faster rate of lung decline, increased rates of exacerbations and shorter survival. By using exome sequencing and extreme phenotype design, it was recently shown that isoforms of dynactin 4 (DCTN4) may influence Pa infection in CF, leading to worse respiratory disease. The purpose of this study was to investigate the role of DCTN4 missense variants on Pa infection incidence, age at first Pa infection and chronic Pa infection incidence in a cohort of adult CF patients from a single centre. Polymerase chain reaction and direct sequencing were used to screen DNA samples for DCTN4 variants. A total of 121 adult CF patients from th

In [ ]:
MAX_CHARS = 1200
text_for_model = doc["text"][:MAX_CHARS]
gold = [m for m in doc["gold_mentions"] if m["end"] <= MAX_CHARS]

print("\nUsed chars:", len(text_for_model))
print("Gold mentions within used text:", len(gold))


MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    device_map="auto",
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
)

generator = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
)


Used chars: 1200
Gold mentions within used text: 42


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

In [ ]:
prompt = f"""
Extract biomedical entity mentions from the text.

Output one entity per line using exactly this format:
mention<TAB>cui<TAB>type

Rules:
- mention must be copied exactly from the text.
- cui must be a UMLS CUI like C0004238 if you know it.
- if you do not know the CUI, write UNKNOWN.
- do not output JSON.
- do not output markdown.
- do not explain.

Text:
{text_for_model}
"""

messages = [
    {
        "role": "system",
        "content": "You extract biomedical entities. Output only TSV lines."
    },
    {
        "role": "user",
        "content": prompt
    },
]

chat_prompt = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
)

out = generator(
    chat_prompt,
    max_new_tokens=180,
    do_sample=False,
    return_full_text=False,
)[0]["generated_text"]

print("\nRaw model output:\n")
print(out)

Both `max_new_tokens` (=180) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Raw model output:

C0004238,DNA,missense,unknown,C0004238,infeciton,age_at_first_Pa_infection,chronic_Pa_infection_incidence


In [ ]:
def norm_mention(s):
    return re.sub(r"\s+", " ", str(s).lower().strip())

def clean_line(line):
    line = line.strip()
    line = re.sub(r"^[\-\*\d\.\)\s]+", "", line)  # 去掉 -、*、1.、2) 等
    line = line.strip("`'\" ")
    return line.strip()

# 建立 mention -> CUI 映射
gold_map = {}
for m in gold:
    key = norm_mention(m["mention"])
    if key not in gold_map:
        gold_map[key] = {
            "mention": m["mention"],
            "cui": m["cui"],
            "type": m["semantic_type"],
        }

pred = []

for line in out.splitlines():
    mention = clean_line(line)

    if not mention:
        continue

    # 跳过废话
    low = mention.lower()
    if low.startswith(("entity", "mention", "output", "text", "biomedical")):
        continue

    # 必须出现在原文中
    if mention.lower() not in text_for_model.lower():
        continue

    key = norm_mention(mention)

    # 如果模型抽到的 mention 正好在 gold 里，补上 CUI
    if key in gold_map:
        pred.append({
            "mention": gold_map[key]["mention"],
            "cui": gold_map[key]["cui"],
            "type": gold_map[key]["type"],
        })
    else:
        # 没对齐到 gold，就保留 UNKNOWN
        pred.append({
            "mention": mention,
            "cui": "UNKNOWN",
            "type": "UNKNOWN",
        })

# 去重
seen = set()
dedup = []
for p in pred:
    k = (norm_mention(p["mention"]), p["cui"])
    if k not in seen:
        seen.add(k)
        dedup.append(p)

pred = dedup

print("\nParsed predictions:")
for p in pred:
    print(p)


Parsed predictions:


In [ ]:
# ============================================================
# UMLS 知识图谱抽取质量评估 Demo
# 只取 MedMentions 1 篇文章，极速可跑完整版本
# ============================================================

import gzip
import re
import itertools
from pathlib import Path

# ----------------------------
# 1. 下载 MedMentions 数据
# ----------------------------

DATA_DIR = Path("/content/medmentions_demo")
DATA_DIR.mkdir(exist_ok=True)

gz_path = DATA_DIR / "corpus_pubtator.txt.gz"

if not gz_path.exists():
    !wget -q -O /content/medmentions_demo/corpus_pubtator.txt.gz https://raw.githubusercontent.com/chanzuckerberg/MedMentions/master/st21pv/data/corpus_pubtator.txt.gz

print("数据下载完成:", gz_path)

# ----------------------------
# 2. 读取第一篇文章
# ----------------------------

def read_first_doc(gz_file):
    block = []

    with gzip.open(gz_file, "rt", encoding="utf-8") as f:
        for line in f:
            line = line.rstrip("\n")
            if line == "":
                if block:
                    break
            else:
                block.append(line)

    title = ""
    abstract = ""
    gold_mentions = []
    pmid = None

    for line in block:
        if "|t|" in line:
            pmid, _, title = line.partition("|t|")
        elif "|a|" in line:
            pmid, _, abstract = line.partition("|a|")
        else:
            parts = line.split("\t")
            if len(parts) >= 6:
                gold_mentions.append({
                    "mention": parts[3],
                    "type": parts[4],
                    "cui": parts[5],
                })

    text = title + " " + abstract
    return pmid, text, gold_mentions


pmid, text, gold_all = read_first_doc(gz_path)

print("\nPMID:", pmid)
print("\n文章文本前 1200 字:")
print(text[:1200])
print("\n原始 Gold UMLS 标注数量:", len(gold_all))

# ----------------------------
# 3. 只取前 1200 个字符
# ----------------------------

MAX_CHARS = 1200
text_for_eval = text[:MAX_CHARS]

gold = [
    m for m in gold_all
    if m["mention"].lower() in text_for_eval.lower()
]

print("\n截断后 Gold 数量:", len(gold))

# ----------------------------
# 4. 工具函数
# ----------------------------

def norm_text(s):
    return re.sub(r"\s+", " ", str(s).lower().strip())

def norm_cui(x):
    """
    MedMentions 里常见格式是 UMLS:C0010674。
    这里统一转成 C0010674。
    """
    x = str(x).strip()
    if x.startswith("UMLS:"):
        x = x.replace("UMLS:", "")
    return x

def is_valid_cui(x):
    x = norm_cui(x)
    return bool(re.fullmatch(r"C\d{7}", x))

def score(pred_set, gold_set):
    tp = len(pred_set & gold_set)
    fp = len(pred_set - gold_set)
    fn = len(gold_set - pred_set)

    precision = tp / (tp + fp) if tp + fp > 0 else 0.0
    recall = tp / (tp + fn) if tp + fn > 0 else 0.0
    f1 = (
        2 * precision * recall / (precision + recall)
        if precision + recall > 0
        else 0.0
    )

    return {
        "TP": tp,
        "FP": fp,
        "FN": fn,
        "Precision": precision,
        "Recall": recall,
        "F1": f1,
    }

def make_edges(cui_set):
    """
    简化知识图谱边：
    同一篇摘要中两个 UMLS CUI 共现，就连一条边。
    """
    cui_list = sorted(set(cui_set))
    edges = set()

    for a, b in itertools.combinations(cui_list, 2):
        edges.add((a, "CO_OCCURS_IN_ABSTRACT", b))

    return edges

# ----------------------------
# 5. 构造一个模拟的大模型抽取结果
# ----------------------------
# 目的：快速测试评估环节。
# 假设模型抽对前 60% 的实体，漏掉后 40%。
# 再加 1 个错误实体，模拟 hallucination / false positive。

gold_valid = [
    {
        "mention": m["mention"],
        "type": m["type"],
        "cui": norm_cui(m["cui"]),
    }
    for m in gold
    if is_valid_cui(m["cui"])
]

keep_n = max(1, int(len(gold_valid) * 0.6))

pred = gold_valid[:keep_n].copy()

pred.append({
    "mention": "disease",
    "type": "T047",
    "cui": "C0012634",
})

print("\n模拟的大模型抽取结果数量:", len(pred))
print("\n模拟的大模型抽取结果:")
for p in pred:
    print(p)

# ----------------------------
# 6. 构造评估集合
# ----------------------------

gold_mention_set = set(
    norm_text(m["mention"])
    for m in gold_valid
)

pred_mention_set = set(
    norm_text(m["mention"])
    for m in pred
)

gold_cui_set = set(
    norm_cui(m["cui"])
    for m in gold_valid
    if is_valid_cui(m["cui"])
)

pred_cui_set = set(
    norm_cui(m["cui"])
    for m in pred
    if is_valid_cui(m["cui"])
)

gold_mention_cui_set = set(
    (norm_text(m["mention"]), norm_cui(m["cui"]))
    for m in gold_valid
    if is_valid_cui(m["cui"])
)

pred_mention_cui_set = set(
    (norm_text(m["mention"]), norm_cui(m["cui"]))
    for m in pred
    if is_valid_cui(m["cui"])
)

gold_edge_set = make_edges(gold_cui_set)
pred_edge_set = make_edges(pred_cui_set)

# ----------------------------
# 7. 计算评估指标
# ----------------------------

results = {
    "Mention exact match": score(pred_mention_set, gold_mention_set),
    "UMLS CUI node match": score(pred_cui_set, gold_cui_set),
    "Mention + CUI match": score(pred_mention_cui_set, gold_mention_cui_set),
    "KG co-occurrence edge match": score(pred_edge_set, gold_edge_set),
}

print("\n==================== 最终评估结果 ====================")

for name, r in results.items():
    print("\n指标:", name)
    print("TP:", r["TP"])
    print("FP:", r["FP"])
    print("FN:", r["FN"])
    print("Precision:", round(r["Precision"], 4))
    print("Recall:", round(r["Recall"], 4))
    print("F1:", round(r["F1"], 4))

# ----------------------------
# 8. 打印 Gold 和 Pred 对比
# ----------------------------

print("\n==================== Gold 标准答案 ====================")
for m in gold_valid:
    print(m["mention"], "|", m["cui"], "|", m["type"])

print("\n==================== 模拟预测结果 ====================")
for m in pred:
    print(m["mention"], "|", m["cui"], "|", m["type"])

print("\n==================== KG 边统计 ====================")
print("Gold CUI 节点数量:", len(gold_cui_set))
print("Pred CUI 节点数量:", len(pred_cui_set))
print("Gold edge 数量:", len(gold_edge_set))
print("Pred edge 数量:", len(pred_edge_set))

print("\nGold edges 前 10 条:")
for e in list(gold_edge_set)[:10]:
    print(e)

print("\nPred edges 前 10 条:")
for e in list(pred_edge_set)[:10]:
    print(e)

数据下载完成: /content/medmentions_demo/corpus_pubtator.txt.gz

PMID: 25763772

文章文本前 1200 字:
DCTN4 as a modifier of chronic Pseudomonas aeruginosa infection in cystic fibrosis Pseudomonas aeruginosa (Pa) infection in cystic fibrosis (CF) patients is associated with worse long-term pulmonary disease and shorter survival, and chronic Pa infection (CPA) is associated with reduced lung function, faster rate of lung decline, increased rates of exacerbations and shorter survival. By using exome sequencing and extreme phenotype design, it was recently shown that isoforms of dynactin 4 (DCTN4) may influence Pa infection in CF, leading to worse respiratory disease. The purpose of this study was to investigate the role of DCTN4 missense variants on Pa infection incidence, age at first Pa infection and chronic Pa infection incidence in a cohort of adult CF patients from a single centre. Polymerase chain reaction and direct sequencing were used to screen DNA samples for DCTN4 variants. A total of 121 a

In [ ]:
# ----------------------------
# 9. Consistency Evaluation
# ----------------------------

from collections import defaultdict

def consistency_by_key(records, key_field, value_field):
    """
    检查同一个 key 是否总是对应同一个 value。
    例如：
    key = mention, value = CUI
    key = CUI, value = type
    """
    mapping = defaultdict(set)

    for r in records:
        key = norm_text(r[key_field]) if key_field == "mention" else r[key_field]
        value = norm_cui(r[value_field]) if value_field == "cui" else r[value_field]
        mapping[key].add(value)

    total_keys = len(mapping)
    inconsistent = {
        k: v for k, v in mapping.items()
        if len(v) > 1
    }

    consistent_keys = total_keys - len(inconsistent)
    consistency_score = consistent_keys / total_keys if total_keys > 0 else 0.0

    return {
        "total_keys": total_keys,
        "consistent_keys": consistent_keys,
        "inconsistent_keys": len(inconsistent),
        "consistency_score": consistency_score,
        "inconsistent_examples": inconsistent,
    }


mention_cui_consistency = consistency_by_key(
    pred,
    key_field="mention",
    value_field="cui"
)

cui_type_consistency = consistency_by_key(
    pred,
    key_field="cui",
    value_field="type"
)

print("\n==================== Consistency Evaluation ====================")

print("\n1. Mention-CUI Consistency")
print("含义：同一个 mention 是否总是链接到同一个 UMLS CUI")
print("Total mentions:", mention_cui_consistency["total_keys"])
print("Consistent mentions:", mention_cui_consistency["consistent_keys"])
print("Inconsistent mentions:", mention_cui_consistency["inconsistent_keys"])
print("Consistency score:", round(mention_cui_consistency["consistency_score"], 4))

if mention_cui_consistency["inconsistent_examples"]:
    print("Inconsistent examples:")
    for k, v in mention_cui_consistency["inconsistent_examples"].items():
        print(k, "->", v)

print("\n2. CUI-Type Consistency")
print("含义：同一个 UMLS CUI 是否总是对应同一个 semantic type")
print("Total CUIs:", cui_type_consistency["total_keys"])
print("Consistent CUIs:", cui_type_consistency["consistent_keys"])
print("Inconsistent CUIs:", cui_type_consistency["inconsistent_keys"])
print("Consistency score:", round(cui_type_consistency["consistency_score"], 4))

if cui_type_consistency["inconsistent_examples"]:
    print("Inconsistent examples:")
    for k, v in cui_type_consistency["inconsistent_examples"].items():
        print(k, "->", v)


==================== Consistency Evaluation ====================

1. Mention-CUI Consistency
含义：同一个 mention 是否总是链接到同一个 UMLS CUI
Total mentions: 26
Consistent mentions: 26
Inconsistent mentions: 0
Consistency score: 1.0

2. CUI-Type Consistency
含义：同一个 UMLS CUI 是否总是对应同一个 semantic type
Total CUIs: 19
Consistent CUIs: 19
Inconsistent CUIs: 0
Consistency score: 1.0


In [ ]:
# ============================================================
# 9. 在已有 pred 后面加入 consistency 冲突，并重新评估一致性
# ============================================================

from collections import defaultdict

# ----------------------------
# 1. 手动加入两个冲突预测
# ----------------------------

# 冲突 1：Mention-CUI 冲突
# 原来 CF -> C0010674
# 现在故意加入 CF -> C1413365
pred.append({
    "mention": "CF",
    "type": "T017",
    "cui": "C1413365",
})

# 冲突 2：CUI-Type 冲突
# 原来 C0010674 -> T038
# 现在故意加入 C0010674 -> T047
pred.append({
    "mention": "cystic fibrosis",
    "type": "T047",
    "cui": "C0010674",
})

print("\n==================== 加入冲突后的 Pred ====================")
for p in pred[-5:]:
    print(p)


# ----------------------------
# 2. Consistency 评估函数
# ----------------------------

def consistency_by_key(records, key_field, value_field):
    """
    检查同一个 key 是否总是对应同一个 value。
    例如：
    key = mention, value = CUI
    key = CUI, value = type
    """
    mapping = defaultdict(set)

    for r in records:
        if key_field == "mention":
            key = norm_text(r[key_field])
        elif key_field == "cui":
            key = norm_cui(r[key_field])
        else:
            key = r[key_field]

        if value_field == "cui":
            value = norm_cui(r[value_field])
        else:
            value = r[value_field]

        mapping[key].add(value)

    inconsistent = {
        k: v for k, v in mapping.items()
        if len(v) > 1
    }

    total_keys = len(mapping)
    inconsistent_keys = len(inconsistent)
    consistent_keys = total_keys - inconsistent_keys
    consistency_score = consistent_keys / total_keys if total_keys > 0 else 0.0

    return {
        "total_keys": total_keys,
        "consistent_keys": consistent_keys,
        "inconsistent_keys": inconsistent_keys,
        "consistency_score": consistency_score,
        "inconsistent_examples": inconsistent,
    }


# ----------------------------
# 3. 计算两个 consistency 指标
# ----------------------------

mention_cui_consistency = consistency_by_key(
    pred,
    key_field="mention",
    value_field="cui"
)

cui_type_consistency = consistency_by_key(
    pred,
    key_field="cui",
    value_field="type"
)


# ----------------------------
# 4. 打印结果
# ----------------------------

print("\n==================== Consistency Evaluation After Conflict ====================")

print("\n1. Mention-CUI Consistency")
print("含义：同一个 mention 是否总是链接到同一个 UMLS CUI")
print("Total mentions:", mention_cui_consistency["total_keys"])
print("Consistent mentions:", mention_cui_consistency["consistent_keys"])
print("Inconsistent mentions:", mention_cui_consistency["inconsistent_keys"])
print("Consistency score:", round(mention_cui_consistency["consistency_score"], 4))

if mention_cui_consistency["inconsistent_examples"]:
    print("\nInconsistent mention examples:")
    for k, v in mention_cui_consistency["inconsistent_examples"].items():
        print(k, "->", v)

print("\n2. CUI-Type Consistency")
print("含义：同一个 UMLS CUI 是否总是对应同一个 semantic type")
print("Total CUIs:", cui_type_consistency["total_keys"])
print("Consistent CUIs:", cui_type_consistency["consistent_keys"])
print("Inconsistent CUIs:", cui_type_consistency["inconsistent_keys"])
print("Consistency score:", round(cui_type_consistency["consistency_score"], 4))

if cui_type_consistency["inconsistent_examples"]:
    print("\nInconsistent CUI examples:")
    for k, v in cui_type_consistency["inconsistent_examples"].items():
        print(k, "->", v)


==================== 加入冲突后的 Pred ====================
{'mention': 'CFTR', 'type': 'T017', 'cui': 'C1413365'}
{'mention': 'pulmonary infection', 'type': 'T038', 'cui': 'C0876973'}
{'mention': 'disease', 'type': 'T047', 'cui': 'C0012634'}
{'mention': 'CF', 'type': 'T017', 'cui': 'C1413365'}
{'mention': 'cystic fibrosis', 'type': 'T047', 'cui': 'C0010674'}

==================== Consistency Evaluation After Conflict ====================

1. Mention-CUI Consistency
含义：同一个 mention 是否总是链接到同一个 UMLS CUI
Total mentions: 26
Consistent mentions: 25
Inconsistent mentions: 1
Consistency score: 0.9615

Inconsistent mention examples:
cf -> {'C1413365', 'C0010674'}

2. CUI-Type Consistency
含义：同一个 UMLS CUI 是否总是对应同一个 semantic type
Total CUIs: 19
Consistent CUIs: 18
Inconsistent CUIs: 1
Consistency score: 0.9474

Inconsistent CUI examples:
C0010674 -> {'T047', 'T038'}


In [ ]:
consistency_results = {
    "Mention-CUI Consistency": mention_cui_consistency,
    "CUI-Type Consistency": cui_type_consistency,
}

print("\n==================== Consistency Score Summary ====================")

for name, r in consistency_results.items():
    print("\n指标:", name)
    print("Total keys:", r["total_keys"])
    print("Consistent keys:", r["consistent_keys"])
    print("Inconsistent keys:", r["inconsistent_keys"])
    print("Consistency score:", round(r["consistency_score"], 4))


==================== Consistency Score Summary ====================

指标: Mention-CUI Consistency
Total keys: 26
Consistent keys: 25
Inconsistent keys: 1
Consistency score: 0.9615

指标: CUI-Type Consistency
Total keys: 19
Consistent keys: 18
Inconsistent keys: 1
Consistency score: 0.9474
